<a href="https://colab.research.google.com/github/BotCalvin/BUS-118S/blob/main/Exercise_2_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exercise 2 — Code Generation with ReACT Prompting
**Tools used:** ChatGPT (ReACT prompting), Google Colab (execution)

You are an AI coding assistant using a ReACT loop (Reason + Act + Observe + Fix).

GOAL:
Generate Python code, then “execute” it in a Colab notebook (the user will run it), observe output, and fix issues iteratively.

ALLOWED TOOLS:
- Python 3 standard library only (no external packages).
- The user will run your code in Google Colab and paste back outputs/errors.

REACT STAGES (follow these headings exactly):
1) REASON/PLAN
- Summarize the problem in 1–2 sentences.
- List assumptions.
- List edge cases.
- Outline steps (3–6 bullets).

2) ACT: GENERATE CODE
- Output a single Python script in one code block.
- Must include:
  - functions with type hints + docstrings
  - input validation + friendly error messages
  - main() + if __name__ == "__main__": main()
- Do NOT include any explanations outside the required stage headings.

3) ACT: TEST PLAN
- Provide 5 tests (inputs + expected outputs), including at least 2 edge cases.

4) OBSERVE
- Ask the user to run the code in Colab and paste the output/error traceback.

5) FIX
- If any failure occurs, explain the cause in 1–3 sentences.
- Provide a revised code block with changes.
- Repeat OBSERVE → FIX until tests pass.

TASK (implement this exactly):
Build a command-line “Grade Calculator” program.
Inputs:
- A list of assignment scores (0–100) entered as a comma-separated string.
- Optional extra credit points (integer, can be negative or positive).
Outputs:
- Mean score (rounded to 2 decimals)
- Letter grade using:
  A: 90–100, B: 80–89.99, C: 70–79.99, D: 60–69.99, F: <60
Rules/Constraints:
- Reject any score outside 0–100.
- If input is blank or malformed, print a clear error and exit with non-zero.
- Final score = mean + extra credit, but clamp final score to [0, 100].
- Show final score and letter grade.
Formatting:
- Print outputs with labels on separate lines.

In [ ]:
from __future__ import annotations

import sys
from typing import List, Tuple


def parse_scores(scores_str: str) -> List[float]:
    """
    Parse a comma-separated list of numeric scores (0-100 inclusive).
    Raises ValueError with a helpful message if parsing fails.
    """
    if scores_str is None or scores_str.strip() == "":
        raise ValueError("Scores input cannot be blank. Example: 88,92,76")

    parts = [p.strip() for p in scores_str.split(",")]
    if any(p == "" for p in parts):
        raise ValueError("Malformed scores list. Remove extra commas and spaces.")

    scores: List[float] = []
    for p in parts:
        try:
            val = float(p)
        except ValueError:
            raise ValueError(f"Invalid score '{p}'. Scores must be numbers 0–100.") from None

        if val < 0 or val > 100:
            raise ValueError(f"Score {val} is out of range. Each score must be 0–100.")
        scores.append(val)

    if len(scores) == 0:
        raise ValueError("No scores provided. Example: 88,92,76")

    return scores


def parse_extra_credit(extra_str: str) -> int:
    """
    Parse extra credit as an integer (can be negative).
    Blank means 0.
    """
    if extra_str is None or extra_str.strip() == "":
        return 0
    try:
        return int(extra_str.strip())
    except ValueError:
        raise ValueError("Extra credit must be an integer (e.g., 0, 2, -3).") from None


def clamp(value: float, lo: float, hi: float) -> float:
    """Clamp value to the inclusive range [lo, hi]."""
    return max(lo, min(hi, value))


def letter_grade(score: float) -> str:
    """Return letter grade based on the given score (0–100)."""
    if score >= 90:
        return "A"
    if score >= 80:
        return "B"
    if score >= 70:
        return "C"
    if score >= 60:
        return "D"
    return "F"


def compute_results(scores: List[float], extra_credit: int) -> Tuple[float, float, str]:
    """
    Compute mean score, final score (mean + extra credit clamped), and letter grade.
    Returns: (mean, final, letter)
    """
    mean = sum(scores) / len(scores)
    final = clamp(mean + extra_credit, 0.0, 100.0)
    return mean, final, letter_grade(final)


def main() -> None:
    """
    CLI entry point.
    Prompts the user for scores and extra credit, prints results.
    Exits non-zero on invalid input.
    """
    try:
        scores_str = input("Enter assignment scores (comma-separated, 0–100): ").strip()
        extra_str = input("Enter extra credit (integer, blank for 0): ").strip()

        scores = parse_scores(scores_str)
        extra = parse_extra_credit(extra_str)

        mean, final, letter = compute_results(scores, extra)

        print(f"Mean score: {mean:.2f}")
        print(f"Final score (clamped): {final:.2f}")
        print(f"Letter grade: {letter}")

    except ValueError as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

Enter assignment scores (comma-separated, 0–100): 70
